# Import Libraries

In [5]:
import os
import cv2
import numpy as np
from tqdm import tqdm
print("All Libraries Loaded")

# Define Paths

In [11]:
# Base Path
BASE_PATH = r"C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET"

# ==========================
# TRAIN PATHS
# ==========================
TRAIN_IMAGE_DIR = os.path.join(BASE_PATH, "Segmentation", "train", "images")
TRAIN_MASK_DIR = os.path.join(BASE_PATH, "Segmentation", "train", "masks")

SAVE_TRAIN_IMAGE_DIR = os.path.join(BASE_PATH, "Preprocessed_segmented", "train", "images")
SAVE_TRAIN_MASK_DIR = os.path.join(BASE_PATH, "Preprocessed_segmented", "train", "masks")

# ==========================
# TEST PATHS
# ==========================
TEST_IMAGE_DIR = os.path.join(BASE_PATH, "Segmentation", "test", "images")
TEST_MASK_DIR = os.path.join(BASE_PATH, "Segmentation", "test", "masks")

SAVE_TEST_IMAGE_DIR = os.path.join(BASE_PATH, "Preprocessed_segmented", "test", "images")
SAVE_TEST_MASK_DIR = os.path.join(BASE_PATH, "Preprocessed_segmented", "test", "masks")

# ==========================
# Create Output Directories
# ==========================
os.makedirs(SAVE_TRAIN_IMAGE_DIR, exist_ok=True)
os.makedirs(SAVE_TRAIN_MASK_DIR, exist_ok=True)

os.makedirs(SAVE_TEST_IMAGE_DIR, exist_ok=True)
os.makedirs(SAVE_TEST_MASK_DIR, exist_ok=True)

print(" Output folders created successfully!")

✅ Output folders created successfully!


# Preprocessing Function

In [12]:
# ==========================
# Image Size
# ==========================
IMG_SIZE = 256


def preprocess_image(image_path):
    """
    Preprocess MRI image
    """

    # Read image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError(f"Could not read image: {image_path}")

    # Gaussian Denoising
    img = cv2.GaussianBlur(img, (3, 3), 0)

    # Resize
    img = cv2.resize(
        img,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_AREA
    )

    # Normalize to [0,1]
    img = img.astype(np.float32) / 255.0

    return img


def preprocess_mask(mask_path):
    """
    Preprocess segmentation mask
    """

    # Read mask
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        raise ValueError(f"Could not read mask: {mask_path}")

    # Resize using nearest-neighbor
    mask = cv2.resize(
        mask,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )

    # Force binary mask (0 and 255 only)
    mask = (mask > 0).astype(np.uint8) * 255

    return mask

# Process Entire DataSet

In [13]:
def process_dataset(image_dir, mask_dir, save_image_dir, save_mask_dir, dataset_name):
    """
    Process an entire dataset (train or test)
    """

    image_files = sorted(os.listdir(image_dir))
    mask_files = sorted(os.listdir(mask_dir))

    print(f"\nProcessing {dataset_name} Dataset")
    print(f"Images : {len(image_files)}")
    print(f"Masks  : {len(mask_files)}")

    assert len(image_files) == len(mask_files), \
        f"{dataset_name}: Image and Mask count mismatch!"

    for img_name, mask_name in tqdm(
            zip(image_files, mask_files),
            total=len(image_files),
            desc=dataset_name):

        img_path = os.path.join(image_dir, img_name)
        mask_path = os.path.join(mask_dir, mask_name)

        # Preprocess
        img = preprocess_image(img_path)
        mask = preprocess_mask(mask_path)

        # Convert normalized image back to uint8 for saving
        save_img = (img * 255).astype(np.uint8)

        # Save image
        cv2.imwrite(
            os.path.join(save_image_dir, img_name),
            save_img
        )

        # Save mask
        cv2.imwrite(
            os.path.join(save_mask_dir, mask_name),
            mask
        )

    print(f"✅ {dataset_name} preprocessing completed.")


# ==========================
# Process Training Dataset
# ==========================
process_dataset(
    TRAIN_IMAGE_DIR,
    TRAIN_MASK_DIR,
    SAVE_TRAIN_IMAGE_DIR,
    SAVE_TRAIN_MASK_DIR,
    "Train"
)

# ==========================
# Process Testing Dataset
# ==========================
process_dataset(
    TEST_IMAGE_DIR,
    TEST_MASK_DIR,
    SAVE_TEST_IMAGE_DIR,
    SAVE_TEST_MASK_DIR,
    "Test"
)

print("\n🎉 All preprocessing completed successfully!")


Processing Train Dataset
Images : 3933
Masks  : 3933


Train: 100%|███████████████████████████████████████████████████████████████████████| 3933/3933 [02:16<00:00, 28.76it/s]


✅ Train preprocessing completed.

Processing Test Dataset
Images : 860
Masks  : 860


Test: 100%|██████████████████████████████████████████████████████████████████████████| 860/860 [00:29<00:00, 29.46it/s]

✅ Test preprocessing completed.

🎉 All preprocessing completed successfully!
